In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

import os
data_path = '/content/drive/MyDrive/f1-telemetry-ml'
labeled_path = f'{data_path}/labeled'

print(f"Labeled data path: {labeled_path}")
print(f"Contents: {os.listdir(labeled_path)}")

In [ ]:

# 1. Load combined globally normalized race labels
labels = pd.read_parquet(f'{labeled_path}/labels_combined.parquet')

# Basic shape of the dataset
print(f"Total labeled corner-instances: {labels.shape[0]}")
print(f"Races included ({labels['race'].nunique()} total): {labels['race'].unique()}")
print(f"Drivers included ({labels['driver'].nunique()} total): {labels['driver'].unique()}")
print("\nCorners per race:")
print(labels.groupby('race')['corner_number'].nunique())

In [ ]:

# 2. Distribution of target scores
labels[['aggression_score', 'line_shape_score', 'oversteer_preference_score']].hist(figsize=(12, 4), bins=30)
plt.suptitle('Distribution of Style Scores (2024 Season - Race Only)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Summary statistics
print("\n=== Target Score Statistics ===")
print(labels[['aggression_score', 'line_shape_score', 'oversteer_preference_score']].describe())

In [ ]:
# 4. Driver style comparison (Turn 1 benchmark)
turn_1 = labels[labels['corner_number'] == 1]

# Define target metric columns
style_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']

# Group by driver, calculate mean across all style metrics, and sort by Aggression
turn_1_driver_styles = (
    turn_1.groupby('driver')[style_cols]
    .mean()
    .sort_values(by='aggression_score', ascending=False)
)

print("\n=== Top Drivers by Turn 1 Style Profile ===")
print(turn_1_driver_styles.head(20).round(4))


In [ ]:
# 5. Session type verification (Should be exclusively 'R')
print("\n=== Session Type Breakdown ===")
print(labels['session_type'].value_counts())

In [ ]:
# 6. Train / Val / Test / Zero-Shot Track Split
output_dir = '../fastf1_data/labeled'

# Define track partitioning
TRAIN_TRACKS = ["Monza", "Monaco", "Silverstone", "Suzuka", "Austin", "Bahrain"]
VAL_TRACKS = ["Singapore", "Austria"]
TEST_IN_DIST_TRACKS = ["Monza", "Silverstone"] 
HOLDOUT_TRACKS = ["Belgium", "Jeddah"]
# 1. Zero-Shot Holdout Set (Unseen Tracks)
zero_shot_df = labels[labels['race'].isin(HOLDOUT_TRACKS)].copy()

# 2. Validation Set (Dedicated Validation Tracks)
val_df = labels[labels['race'].isin(VAL_TRACKS)].copy()

# 3. Pure Training-Only Tracks (Monaco, Suzuka, Austin, Bahrain)
pure_train_tracks = list(set(TRAIN_TRACKS) - set(TEST_IN_DIST_TRACKS))
pure_train_df = labels[labels['race'].isin(pure_train_tracks)].copy()

# 4. Handle In-Distribution Overlapping Tracks (Monza & Silverstone)
# Split by lap number to prevent corner leakage between Train and In-Dist Test
in_dist_df = labels[labels['race'].isin(TEST_IN_DIST_TRACKS)].copy()

train_overlap_list = []
test_in_dist_list = []

np.random.seed(42)  # For reproducible lap splits
for (race, driver), group in in_dist_df.groupby(['race', 'driver']):
    unique_laps = group['lap_number'].unique()
    np.random.shuffle(unique_laps)
    
    # 80% laps for training, 20% laps for in-distribution testing
    split_idx = int(len(unique_laps) * 0.8)
    train_laps = unique_laps[:split_idx]
    test_laps = unique_laps[split_idx:]
    
    train_overlap_list.append(group[group['lap_number'].isin(train_laps)])
    test_in_dist_list.append(group[group['lap_number'].isin(test_laps)])

train_overlap_df = pd.concat(train_overlap_list, ignore_index=True)
test_in_dist_df = pd.concat(test_in_dist_list, ignore_index=True)

# Combine pure training tracks with the training lap slice of overlap tracks
train_df = pd.concat([pure_train_df, train_overlap_df], ignore_index=True)

# ---------------------------------------------------------
# Save Processed Parquet Splits
# ---------------------------------------------------------
train_df.to_parquet(f'{output_dir}/train_data.parquet', index=False)
val_df.to_parquet(f'{output_dir}/val_data.parquet', index=False)
test_in_dist_df.to_parquet(f'{output_dir}/test_in_dist_data.parquet', index=False)
zero_shot_df.to_parquet(f'{output_dir}/test_zero_shot.parquet', index=False)

# ---------------------------------------------------------
# Summary Verification Printout
# ---------------------------------------------------------
print("=== Data Split Summary ===")
print(f"Train Set           ({train_df['race'].nunique()} tracks): {train_df.shape[0]} rows -> {train_df['race'].unique().tolist()}")
print(f"Val Set             ({val_df['race'].nunique()} tracks): {val_df.shape[0]} rows -> {val_df['race'].unique().tolist()}")
print(f"In-Dist Test Set    ({test_in_dist_df['race'].nunique()} tracks): {test_in_dist_df.shape[0]} rows -> {test_in_dist_df['race'].unique().tolist()}")
print(f"Zero-Shot Test Set  ({zero_shot_df['race'].nunique()} tracks): {zero_shot_df.shape[0]} rows -> {zero_shot_df['race'].unique().tolist()}")